# Monthly heating + cooling load profile


Typical NL year of district heat/cool demand vs. system capacity. Validates that the system can meet demand month-by-month with the planned mix.

## Output
- `05_visualizations/charts/monthly_load_profile_v01.png`


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
heat_demand = np.array([10.0, 9.5, 7.5, 5.0, 2.5, 1.5, 1.0, 1.0, 2.0, 5.0, 8.0, 9.5])
cool_demand = np.array([0, 0, 0.5, 1.5, 3.0, 4.5, 5.0, 5.0, 4.0, 1.5, 0, 0])
print(f"Annual heating demand (integrated): {heat_demand.sum() * 730 / 1000:.1f} GWh")  # ~730 hr/month
print(f"Annual cooling demand: {cool_demand.sum() * 730 / 1000:.1f} GWh")

> **Note on annual energy basis.** The `heat_demand`/`cool_demand` arrays are *peak monthly loads* (MW). Integrating them as `Σ(MW)·730 h` gives an **upper bound** (~45.6 GWh heating) that assumes every month runs at its peak all month — it is *not* the delivered-energy figure used in the economics. The LCoE model and energy balance use full-load-equivalent hours (2,400 h heating, 1,500 h cooling), i.e. 10 MW × 2,400 = 24,000 MWh heating + 5 MW × 1,500 = 7,500 MWh cooling = **31,500 MWh/yr delivered**. Use the FLEH-based 31,500 MWh for any cost-per-MWh calculation; this chart is for month-by-month capacity adequacy only.

## Heat supply mix (winter dominates)

Stack: geothermal direct → HP-1 (return brine) → HP-2 (ATES warm well). Combined capacity 10.7 MWth meets the 10 MWth peak in January with 7% headroom.

## Cool supply mix (summer dominates)

Absorption chiller (geothermal-driven) handles 3 MWth, ATES cold well 1 MWth, electric chiller 2 MWth for peaks. The 6 MWth combined gives 20% headroom over the 5 MWth target.

In [ ]:
# --- Monthly demand vs installed capacity ---
import sys
sys.path.insert(0, '../../../03_analysis/scripts')
import matplotlib.pyplot as plt
from plotting_utils import apply_style, COLORS
apply_style()

x = np.arange(len(months))
fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x, heat_demand, color=COLORS['producer'], alpha=0.85, label='Heating demand (MWth)')
ax.bar(x, -cool_demand, color=COLORS['chiller'], alpha=0.85, label='Cooling demand (MWth)')
ax.axhline(10.7, color=COLORS['hp'], ls='--', lw=1.2, label='Heating capacity 10.7 MWth')
ax.axhline(-6.0, color='#0F6E56', ls='--', lw=1.2, label='Cooling capacity 6.0 MWth')
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(months)
ax.set_ylabel('Thermal load (MWth)    heating +  /  cooling \u2212')
ax.set_title('Monthly heating & cooling demand vs installed capacity', loc='left', fontweight='bold')
ax.legend(loc='upper right', ncol=2, fontsize=8)
plt.tight_layout()
plt.savefig('../../../05_visualizations/charts/monthly_load_profile_v01.png', dpi=150, bbox_inches='tight')
plt.savefig('../../../05_visualizations/charts/monthly_load_profile_v01.svg', bbox_inches='tight')
plt.show()

In [ ]:
# --- Annual energy balance (FLEH basis) ---
labels = ['Heating\ndelivered', 'Cooling\ndelivered', 'Electricity\ninput']
vals = [10 * 2400, 5 * 1500, 8750]          # 24,000 + 7,500 MWh delivered; 8,750 MWh elec
colors = [COLORS['producer'], COLORS['chiller'], COLORS['hp']]
fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.bar(labels, vals, color=colors, alpha=0.85)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v + 400, f'{v:,} MWh', ha='center', fontsize=10, fontweight='bold')
spf = (vals[0] + vals[1]) / vals[2]
ax.set_ylabel('Energy (MWh/yr)')
ax.set_title(f'Annual energy balance  \u2014  SPF = {spf:.1f}   (net CO\u2082 avoided \u2248 2,600 t/yr)',
             loc='left', fontweight='bold')
plt.tight_layout()
plt.savefig('../../../05_visualizations/charts/energy_balance_v01.png', dpi=150, bbox_inches='tight')
plt.savefig('../../../05_visualizations/charts/energy_balance_v01.svg', bbox_inches='tight')
plt.show()

In [ ]:
# --- Hybrid surface-system process flow diagram (schematic) ---
import matplotlib.patches as mpatches
fig, ax = plt.subplots(figsize=(12, 7))
ax.set_xlim(0, 12); ax.set_ylim(0, 8); ax.axis('off')

def _box(x, y, w, h, text, color):
    ax.add_patch(mpatches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.02',
                 fc=color, ec='black', alpha=0.85))
    ax.text(x + w / 2, y + h / 2, text, ha='center', va='center', fontsize=9, fontweight='bold')

def _arrow(x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle='->', lw=1.5))

_box(0.3, 3.5, 2.0, 1.2, 'Geothermal\ndoublet\n5.1 MWth', COLORS['producer'])
_box(3.0, 3.5, 1.8, 1.2, 'HX-1\nbrine \u2192 clean', COLORS['neutral'])
_box(5.5, 5.2, 1.8, 1.1, 'HP-1\n+3.4 MWth', COLORS['hp'])
_box(5.5, 3.5, 1.8, 1.1, 'District\nheating loop', COLORS['producer'])
_box(5.5, 1.8, 1.8, 1.1, 'HP-2 / ATES\n+2.2 MWth', COLORS['hp'])
_box(8.2, 5.2, 1.8, 1.1, 'Absorption\nchiller 3 MW', COLORS['chiller'])
_box(8.2, 3.5, 1.8, 1.1, 'Electric\nchiller 2 MW', COLORS['chiller'])
_box(8.2, 1.8, 1.8, 1.1, 'ATES\nstorage', COLORS['injector'])
_arrow(2.3, 4.1, 3.0, 4.1)
_arrow(4.8, 4.1, 5.5, 4.1)
_arrow(4.8, 4.1, 5.5, 5.7); _arrow(4.8, 4.1, 5.5, 2.35)
_arrow(7.3, 5.7, 8.2, 5.7); _arrow(7.3, 4.1, 8.2, 4.1); _arrow(7.3, 2.35, 8.2, 2.35)
ax.set_title('Hybrid surface system \u2014 process flow', fontsize=13, fontweight='bold', loc='left')
plt.tight_layout()
plt.savefig('../../../05_visualizations/system_diagrams/process_flow_diagram_v01.svg', bbox_inches='tight')
plt.savefig('../../../05_visualizations/system_diagrams/process_flow_diagram_v01.png', dpi=150, bbox_inches='tight')
plt.show()